In [ ]:
import pandas as pd

df = pd.read_csv("zone_divided_data.csv")

# listing all non-antibiotic columns to skip
FIXED_COLUMNS = {
    "Patient Name",
    "UMR No.",
    "Reports Photo",
    "Bill No.",
    "Age",
    "Gender",
    "Date",
    "Gram Stain",
    "Organism Isolated",
    "Specimen Type",
    "Department",
    "Colony Count",
    "patient_id",   # if present
    "geo_zone",
}

def get_antibiotic_col(name: str) -> str | None:
    """
    Case-insensitive match to find the correct antibiotic column.
    """
    for col in df.columns:
        if col in FIXED_COLUMNS:
            continue
        if col.lower() == name.lower():
            return col
    return None

def is_patient_resistant(pid: int, ab_col: str) -> bool:
    """
    True if any record for this UMR No. has 'r' in that antibiotic column.
    """
    subset = df[df["UMR No."] == pid]
    if subset.empty:
        return False
    return subset[ab_col].astype(str).str.strip().str.lower().eq("r").any()

def get_patient_zones(pid: int) -> list[str]:
    """
    All unique geo_zones where this UMR No. appears.
    """
    return df[df["UMR No."] == pid]["geo_zone"].dropna().unique().tolist()

def is_zone_resistant(zone: str, ab_col: str) -> bool:
    """
    True if any record in this geo_zone has 'R' for the antibiotic.
    """
    subset = df[df["geo_zone"] == zone]
    if subset.empty:
        return False
    return subset[ab_col].astype(str).str.strip().str.lower().eq("r").any()

def get_resistant_zones_for_patient(pid: int, ab_col: str) -> list[str]:
    """
    Of all zones for the patient, which show resistance?
    """
    return [z for z in get_patient_zones(pid) if is_zone_resistant(z, ab_col)]

def get_resistant_departments(ab_col: str) -> list[str]:
    """
    Which departments have any 'R' for this antibiotic?
    """
    return (
        df[df[ab_col].astype(str).str.strip().str.lower().eq("r")]
        ["Department"]
        .dropna()
        .unique()
        .tolist()
    )

def check_resistance(pid: int, antibiotic_name: str) -> None:
    """
    Cascade:
      1) Patient
      2) All of patient's zones
      3) Hospital departments
    """
    ab_col = get_antibiotic_col(antibiotic_name)
    if not ab_col:
        print(f"❌ Antibiotic '{antibiotic_name}' not found in dataset.")
        return

    # Check 1 - Patient level
    if is_patient_resistant(pid, ab_col):
        print(f"🚨 Patient (UMR No. {pid}) is resistant to {antibiotic_name}.")
        # return

    # Check 2 - Geographic Zone level (all zones for that patient)
    zones = get_resistant_zones_for_patient(pid, ab_col)
    if zones:
        print(
            f"⚠️ Resistance to {antibiotic_name} detected in zone(s) "
            f"{', '.join(zones)} for patient {pid}."
        )
        # return

    # 3 - Department level
    depts = get_resistant_departments(ab_col)
    if depts:
        print(
            f"ℹ️ No resistance at patient or zone level, but resistance exists in "
            f"department(s): {', '.join(depts)}."
        )
    else:
        print(
            f"✅ No resistance detected at patient, zone, or department level "
            f"for {antibiotic_name}."
        )

if __name__ == "__main__":
    try:
        pid_input = int(input("Enter UMR No. (patient ID): ").strip())
    except ValueError:
        print("❌ Invalid UMR No. Please enter a number.")
    else:
        ab_input = input("Enter antibiotic name: ").strip()
        check_resistance(pid_input, ab_input)


ℹ️ No resistance at patient or zone level, but resistance exists in department(s): obgy, general medicine, orthopaedics, urology.
